# Выбор генеративной модели

В блокноте сравниваются локальные языковые модели для генерации ответа по найденным фрагментам регламента. Главный критерий — понимание перефразированных вопросов: вопрос сформулирован другими словами, чем текст регламента, и модель должна понять, что фрагмент отвечает на него. Второй критерий — модель не должна выдумывать ответ, когда его нет во фрагментах (вопросы-ловушки).

Поиск во всех экспериментах одинаковый — лучший вариант из блокнота `parameter_tuning.ipynb`: гибрид RRF с переранжированием. Меняется только генеративная модель, поэтому разница в ответах объясняется моделью, а не поиском.

**Порядок работы:**

1. каждая модель отвечает на одни и те же 36 вопросов: 16 перефразированных, все 25 вопросов ТЗ (часть из них совпадает) и 3 ловушки; прогон повторяется при 3, 5 и 10 фрагментах в контексте;
2. автоматически считаются доля распознанных ловушек, доля ложных отказов, наличие ссылок на фрагменты и время ответа;
3. ответы размечаются по метрикам ТЗ: точность ответа 0/1/2 и наличие галлюцинаций; модели сравниваются при 5 фрагментах;
4. сравнивается качество ответов при 3, 5 и 10 фрагментах, как требует ТЗ.

**Подготовка.** Модели запускаются через [Ollama](https://ollama.com). Нужно установить Ollama, затем скачать модели командами `ollama pull <модель>` из списка ниже. Модели 7–8B в квантовании Q4 занимают около 5 ГБ и работают на процессоре при 16 ГБ оперативной памяти, но медленно: около 1–2 минут на вопрос. Поэтому прогон выполнялся в Google Colab на видеокарте T4, где ответ занимает несколько секунд. Ответы сохраняются в `data/generation/answers.csv` по мере получения, поэтому прерванный прогон продолжается с того же места.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluate import contains, load_queries
from src.generate import answer, source_label
from src.hybrid_retrieval import build_hybrid

BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#9a9a9a"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c9c9c4",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e1",
    "axes.axisbelow": True, "font.size": 10,
})

OUT_DIR = ROOT / "data" / "generation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANSWERS_PATH = OUT_DIR / "answers.csv"

## 1. Модели и вопросы

Сравниваются модели одного класса — 3–8 млрд параметров, которые помещаются в 16 ГБ оперативной памяти:

- **Qwen2.5 7B** — многоязычная модель Alibaba, одна из сильнейших открытых моделей своего размера на русском языке;
- **Gemma 3 4B** — компактная многоязычная модель Google, проверяет, насколько хватает модели вдвое меньше;
- **YandexGPT 5 Lite 8B** — модель Яндекса, обученная в основном на русских текстах.

Список можно изменить: достаточно поправить `MODELS` и скачать модель через `ollama pull`.

In [ ]:
MODELS = {
    "Qwen2.5 7B": "qwen2.5:7b",
    "Gemma 3 4B": "gemma3:4b",
    "YandexGPT 5 Lite 8B": "hf.co/yandex/YandexGPT-5-Lite-8B-instruct-GGUF",
}

installed = {m["name"] for m in requests.get("http://localhost:11434/api/tags", timeout=10).json()["models"]}
for label, name in MODELS.items():
    ok = any(i == name or i.split(":")[0] == name.split(":")[0] for i in installed)
    print(f"{label:22s} {name:50s} {'скачана' if ok else 'НЕТ: ollama pull ' + name}")

In [ ]:
queries = [q for q in load_queries() if q["category"] in ("paraphrase", "trap") or q.get("tz")]
print(f"вопросов: {len(queries)}")
pd.Series([q["category"] for q in queries]).value_counts().rename("вопросов").to_frame()

In [ ]:
retriever = build_hybrid("rrf", rerank=True)

## 2. Прогон моделей

Каждая модель отвечает на все вопросы при 3, 5 и 10 фрагментах в контексте. Для 10 фрагментов контекстное окно увеличено до 16 тыс. токенов.

Для каждого вопроса сохраняются ответ, источники, время генерации и признак, есть ли ответ в переданных фрагментах (`context_has_answer`). Этот признак показывает, чья ошибка: если правильного фрагмента в контексте нет, модель не могла ответить, и это ошибка поиска, а не генерации.

In [ ]:
COLUMNS = ["model", "k", "query_id", "tz", "category", "question", "answer", "sources",
           "context_has_answer", "seconds", "accuracy", "hallucination", "note"]


def load_answers() -> pd.DataFrame:
    """Читает сохранённые ответы; если файла ещё нет, возвращает пустую таблицу."""
    if ANSWERS_PATH.exists():
        return pd.read_csv(ANSWERS_PATH, dtype={"tz": str}, keep_default_na=False)
    return pd.DataFrame(columns=COLUMNS)


def run(model_label: str, k: int = 5, num_ctx: int = 8192) -> None:
    """Получает ответы модели на все вопросы и дописывает их в answers.csv.

    Уже полученные ответы пропускаются; вопрос, на котором модель вернула ошибку, будет повторён при следующем запуске.
    """
    done = load_answers()
    done_ids = set(done.loc[(done["model"] == model_label) & (done["k"].astype(int) == k), "query_id"])
    todo = [q for q in queries if q["query_id"] not in done_ids]
    for i, q in enumerate(todo, start=1):
        try:
            result = answer(q["query"], retriever, MODELS[model_label], k, num_ctx)
        except (RuntimeError, requests.ConnectionError) as error:
            print(f"{model_label} k={k}: {q['query_id']} пропущен, {error}")
            continue
        texts = [hit.chunk["text"] for hit in result.sources]
        row = {
            "model": model_label, "k": k, "query_id": q["query_id"], "tz": q.get("tz", ""),
            "category": q["category"], "question": q["query"], "answer": result.text,
            "sources": " | ".join(source_label(hit) for hit in result.sources),
            "context_has_answer": int(any(contains(t, e) for t in texts for e in q["evidence"])),
            "seconds": round(result.seconds, 1), "accuracy": "", "hallucination": "", "note": "",
        }
        pd.DataFrame([row], columns=COLUMNS).to_csv(
            ANSWERS_PATH, mode="a", header=not ANSWERS_PATH.exists(), index=False, encoding="utf-8")
        print(f"{model_label} k={k}: {i}/{len(todo)} {q['query_id']} {result.seconds:.0f} с")

In [ ]:
for label in MODELS:
    for k in (3, 5, 10):
        run(label, k=k, num_ctx=16384 if k == 10 else 8192)

## 3. Автоматические показатели

Без ручной разметки можно посчитать:

- **ловушки распознаны** — доля ловушек, на которые модель отказалась отвечать;
- **ложные отказы** — доля вопросов с ответом в контексте, на которые модель всё равно отказалась отвечать;
- **ответы со ссылками** — доля ответов, где есть ссылка на фрагмент вида [1], как требует промпт;
- **время, с** — медианное время генерации; медиана, а не среднее, потому что первый ответ каждой модели включает её загрузку в память.

Отказом считается ответ с фразой «не найден», «нет информации», «не указано» или «нет прямого ответа»: модели не всегда повторяют фразу из промпта дословно, а отказываются своими словами.

In [ ]:
REFUSAL = "не найден|нет информации|не указан|нет прямого ответа"


def load_typed() -> pd.DataFrame:
    """Читает ответы и приводит столбцы к числам; неразмеченные оценки становятся NaN."""
    df = load_answers()
    for column in ["k", "context_has_answer"]:
        df[column] = df[column].astype(int)
    for column in ["seconds", "accuracy", "hallucination"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    df["refused"] = df["answer"].str.lower().str.contains(REFUSAL).astype(int)
    df["cited"] = df["answer"].str.contains(r"\[\d+\]").astype(int)
    return df


def auto_metrics(df: pd.DataFrame) -> pd.Series:
    """Считает показатели одной модели, не требующие ручной разметки."""
    traps = df[df["category"] == "trap"]
    answerable = df[(df["category"] != "trap") & (df["context_has_answer"] == 1)]
    return pd.Series({
        "ловушки распознаны": traps["refused"].mean(),
        "ложные отказы": answerable["refused"].mean(),
        "ответы со ссылками": df.loc[df["category"] != "trap", "cited"].mean(),
        "время, с": df["seconds"].median(),
    })


answers = load_typed()
k5 = answers[answers["k"] == 5]
k5.groupby("model").apply(auto_metrics).reindex(list(MODELS)).round(3)

In [ ]:
print("Доля вопросов, у которых ответ есть в 5 переданных фрагментах:")
answerable = k5[k5["category"] != "trap"].drop_duplicates("query_id")
answerable.groupby("category")["context_has_answer"].mean().round(2)

## 4. Разметка ответов

Метрики ТЗ «точность ответа» и «доля галлюцинаций» требуют оценки каждого ответа. В `data/generation/answers.csv` для каждой строки заполнены три столбца:

- `accuracy`: **2** — ответ верный и полный, **1** — частично верный, **0** — неверный или нет ответа; для ловушки 2 — модель отказалась отвечать, 0 — выдумала ответ;
- `hallucination`: **1** — в ответе есть факт, которого нет в переданных фрагментах, **0** — нет;
- `note` — краткое обоснование оценки.

Ответы сверялись с фразами-доказательствами из `data/queries.jsonl` и с текстом переданных фрагментов (столбец `sources`). Разметка выполнена с помощью LLM и выборочно проверена вручную.

In [ ]:
marked = answers.dropna(subset=["accuracy", "hallucination"]).copy()
print(f"размечено: {len(marked)} из {len(answers)}")
marked["paraphrase"] = marked["category"] == "paraphrase"


def manual_metrics(df: pd.DataFrame) -> pd.Series:
    """Считает точность и долю галлюцинаций одной модели по разметке."""
    para = df[df["paraphrase"]]
    return pd.Series({
        "точность, все (0–2)": df["accuracy"].mean(),
        "верных ответов, все": (df["accuracy"] == 2).mean(),
        "точность, перефразы (0–2)": para["accuracy"].mean(),
        "верных ответов, перефразы": (para["accuracy"] == 2).mean(),
        "галлюцинации": df["hallucination"].mean(),
    })


manual = marked[marked["k"] == 5].groupby("model").apply(manual_metrics).reindex(list(MODELS)).dropna(how="all")
manual.round(3) if len(manual) else "Ответы ещё не размечены: заполните accuracy и hallucination в answers.csv"

In [ ]:
def plot_manual(manual: pd.DataFrame) -> None:
    """Рисует долю верных ответов и долю галлюцинаций по моделям."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    fig.suptitle("Качество ответов моделей (5 фрагментов в контексте)", x=0.01, ha="left", fontsize=12)
    y = np.arange(len(manual))
    axes[0].barh(y - 0.2, manual["верных ответов, все"], 0.4, color=BLUE, label="все вопросы")
    axes[0].barh(y + 0.2, manual["верных ответов, перефразы"], 0.4, color=ORANGE, label="перефразы")
    axes[0].set_title("Доля верных ответов", loc="left", fontsize=10, color="#52514e")
    axes[1].barh(y, manual["галлюцинации"], 0.5, color=GREY)
    axes[1].set_title("Доля ответов с галлюцинациями", loc="left", fontsize=10, color="#52514e")
    for ax in axes:
        ax.set_yticks(y, manual.index)
        ax.set_xlim(0, 1)
    axes[0].invert_yaxis()
    axes[1].set_yticklabels([])
    fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncol=2, frameon=False)
    plt.tight_layout()


if len(manual):
    plot_manual(manual)
else:
    print("Сначала разметьте ответы")

**Вывод.** Лучше всех перефразированные вопросы понимает YandexGPT 5 Lite 8B: 88 % верных ответов на перефразах против 62 % у Qwen2.5 7B и 69 % у Gemma 3 4B, по всем вопросам — 94 % против 78 % и 69 %. Все три ловушки YandexGPT распознала, хотя отказывается своими словами, а не фразой из промпта. Её недостатки — она не ставит ссылки на фрагменты и иногда достраивает ответ сверх текста: галлюцинации есть в 3 ответах из 36, причём в одном случае модель прямо предупредила, что ответа в регламенте нет, и дала предположение. Ответ генерируется примерно вдвое дольше, чем у Qwen, но несколько секунд на GPU для справочной системы приемлемы.

Qwen2.5 7B почти не выдумывает (одна галлюцинация) и распознала все ловушки, но часто ошибается иначе: вместо ответа выдаёт только номер фрагмента «[1]» или отказывается, хотя ответ есть в контексте. Gemma 3 4B — самая быстрая, но на одну ловушку выдумала ответ, часто копирует фрагмент целиком вместе с подписью вместо ответа и ставит ссылки списком в начале.

По главному критерию — пониманию перефразов — выбирается YandexGPT 5 Lite 8B.

## 5. Число фрагментов в контексте

ТЗ требует сравнить качество ответов при 3, 5 и 10 фрагментах. Больше фрагментов — выше шанс, что ответ есть в контексте, но модели сложнее найти его среди лишнего текста, и ответ генерируется дольше. Сравнение сделано для всех трёх моделей: так видно, зависит ли влияние объёма контекста от модели.

In [ ]:
answers = load_typed()
marked = answers.dropna(subset=["accuracy", "hallucination"])
by_k = answers[answers["category"] != "trap"].groupby(["model", "k"])[["context_has_answer"]].mean().rename(
    columns={"context_has_answer": "ответ есть в контексте"})
by_k["время, с"] = answers.groupby(["model", "k"])["seconds"].median()
by_k = by_k.join(marked.groupby(["model", "k"]).agg(
    **{"точность (0–2)": ("accuracy", "mean"),
       "верных ответов": ("accuracy", lambda s: (s == 2).mean()),
       "галлюцинации": ("hallucination", "mean")}))
by_k = by_k.join(marked[marked["category"] == "paraphrase"].groupby(["model", "k"])["accuracy"].apply(
    lambda s: (s == 2).mean()).rename("верных на перефразах"))
by_k.round(3)

In [ ]:
def plot_by_k(by_k: pd.DataFrame) -> None:
    """Рисует долю верных ответов при 3, 5 и 10 фрагментах для каждой модели."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
    fig.suptitle("Качество ответов в зависимости от числа фрагментов", x=0.01, ha="left", fontsize=12)
    for (label, color) in zip(MODELS, [BLUE, ORANGE, AQUA]):
        if label not in by_k.index.get_level_values("model"):
            continue
        part = by_k.loc[label]
        axes[0].plot(part.index, part["верных ответов"], "o-", color=color, label=label)
        axes[1].plot(part.index, part["верных на перефразах"], "o-", color=color)
    axes[0].set_title("Доля верных ответов, все вопросы", loc="left", fontsize=10, color="#52514e")
    axes[1].set_title("Доля верных ответов, перефразы", loc="left", fontsize=10, color="#52514e")
    for ax in axes:
        ax.set_xticks([3, 5, 10])
        ax.set_xlabel("фрагментов в контексте")
        ax.set_ylim(0, 1)
    fig.legend(loc="upper right", ncol=3, frameon=False)
    plt.tight_layout()


if by_k["верных ответов"].notna().any():
    plot_by_k(by_k)
else:
    print("Сначала разметьте ответы")

**Вывод.** _Заполняется после прогона k=3 и k=10 для всех моделей и разметки новых ответов: при каком числе фрагментов ответы лучше и стоит ли увеличение контекста потраченного времени. По Qwen2.5 7B уже видно, что доля верных ответов при 3, 5 и 10 фрагментах почти одинакова (78 %, 78 %, 75 %), хотя при 10 фрагментах ответ есть в контексте всегда: ошибки Qwen связаны не с поиском, а с тем, как модель формулирует ответ._

## Итог

_Заполняется в конце: выбранная модель, число фрагментов и обоснование._